<!-- nav -->
[← 8.03 · Python RegExp](08_03_Python_RegExp.ipynb) · [Home](../../README.md) · [8.05 · TypeScript Patterns →](08_05_TypeScript_Patterns.ipynb)

# 8.04 · TypeScript Types

TypeScript writes its types in a grammar of their own, beside its expressions:

```text
let reading: { id: number; celsius?: number } = parse(text) as Reading;
type Readonly<T> = { readonly [K in keyof T]: T[K] };
```

`{ id: number; celsius?: number }`, `Reading` and `{ readonly [K in keyof T]: T[K] }` are types: an object type, a
reference to a named type, a mapped type. `TypeScript Types` is their dialect, its 37 kinds typescript-estree's type
nodes, named and shaped as mbse-programs has them
([CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md)).
Its `Term` lists, beside them, the six TypeScript5 terms a type may hold, a literal's among them, as in the literal
type `"on"`. A type stands where a declaration annotates, mbse-programs', and in TypeScript5's expressions: after `as`
and `satisfies`, in `<T>x`, in a function's annotations and type arguments (tutorial 8, 08.1).

This page builds types from a keyword up, in the builders' fluent calls, prints each as TypeScript writes it, finds
where TypeScript5's terms hold one, shows what a type binds, validates them, and reads the editions that have its kinds.
Like the rest of Part I, it starts from `dialects()`; `typed` holds a type's spec in the dialect's holder,
`S.TypeScript.Types.Expression`, `reference` writes a reference to a named type, and `keyword` a keyword type. No reader
here reads TypeScript's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.TypeScript import TypeScript5, Types as TypeScriptTypes
from toolkit import dialects
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = dialects()
S = store.Expressions
TYPES = TypeScriptTypes.Expressions.DIALECT
ToText = TypeScriptTypes.Text.ToText
reference = lambda name: lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name(name))))
keyword = lambda tag: lambda t: getattr(t, tag)(lambda k: k)  # a keyword type: `string` is tsstringkeyword


def typed(spec):
    """The type `spec` writes, held by its dialect's holder, written in a transaction of its own."""
    with store.transaction():
        return S.TypeScript.Types.Expression(lambda e: e.root(spec))


def show(held):
    """A type's text, as TypeScript writes it, and its kind."""
    print(f"{ToText(held):36}", Terms.kind_of(Terms.term(held))[0])

<a id="T08.04.1"></a>
## 08.04.1 · Keywords, references and arrays

The smallest types are keywords, `string`, `number`, `boolean`, `unknown`, each a kind of its own with nothing in it,
`tsstringkeyword` and the rest. A named type, `Reading` or `T`, is a `tstypereference`, its `typeName` an `identifier`,
and, where it is generic, its `typeArguments`, `Array<string>`. `string[]` is a `tsarraytype` of its `elementType`:

In [2]:
specs = [
    keyword("tsstringkeyword"), keyword("tsnumberkeyword"), keyword("tsunknownkeyword"),
    reference("Reading"),
    lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("Array")))
        .typeArguments(lambda a: a.params([keyword("tsstringkeyword")]))),
    lambda t: t.tsarraytype(lambda a: a.elementType(keyword("tsstringkeyword"))),
]
for spec in specs:
    show(typed(spec))

string                               tsstringkeyword
number                               tsnumberkeyword
unknown                              tsunknownkeyword
Reading                              tstypereference
Array<string>                        tstypereference
string[]                             tsarraytype


<a id="T08.04.2"></a>
## 08.04.2 · Unions, literal types, object types and function types

Types combine. A union, `string | number`, is a `tsuniontype` of its `types`; a literal type, `"on"`, a `tsliteraltype`
holding TypeScript5's literal; an object type, `{ id: number; celsius?: number }`, a `tstypeliteral` of its `members`,
each a helper only types hold, here a `tspropertysignature`, its `key`, `optional` and annotation; a function type,
`(x: number) => string`, a `tsfunctiontype`, its parameters `TypeScript Patterns`' terms (the next page), its
return type an annotation. `keyof T` is a `tstypeoperator`, `T[K]` a `tsindexedaccesstype`. Each nests the types of
08.04.1, and the printer writes parentheses where TypeScript's precedence needs them:

In [3]:
annotation = lambda spec: lambda a: a.typeAnnotation(spec)  # `: spec`, an annotation
on = lambda t: t.tsliteraltype(lambda q: q.literal(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar("on")).quote('"'))))
off = lambda t: t.tsliteraltype(lambda q: q.literal(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar("off")).quote('"'))))
function = lambda t: t.tsfunctiontype(lambda f: f
    .params([lambda p: p.identifier(lambda i: i.name("x").typeAnnotation(annotation(keyword("tsnumberkeyword"))))])
    .returnType(annotation(keyword("tsstringkeyword"))))
specs = [
    lambda t: t.tsuniontype(lambda u: u.types([keyword("tsstringkeyword"), keyword("tsnumberkeyword")])),
    lambda t: t.tsuniontype(lambda u: u.types([on, off])),
    lambda t: t.tstypeliteral(lambda o: o.members([
        lambda m: m.tspropertysignature(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("id")))
            .typeAnnotation(annotation(keyword("tsnumberkeyword")))),
        lambda m: m.tspropertysignature(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("celsius"))).optional(True)
            .typeAnnotation(annotation(keyword("tsnumberkeyword"))))])),
    function,
    lambda t: t.tsuniontype(lambda u: u.types([function, keyword("tsnullkeyword")])),
    lambda t: t.tstypeoperator(lambda o: o.operator("keyof").typeAnnotation(reference("T"))),
    lambda t: t.tsindexedaccesstype(lambda a: a.objectType(reference("T")).indexType(reference("K"))),
]
for spec in specs:
    show(typed(spec))

string | number                      tsuniontype
"on" | "off"                         tsuniontype
{ id: number; celsius?: number; }    tstypeliteral
(x: number) => string                tsfunctiontype
((x: number) => string) | null       tsuniontype
keyof T                              tstypeoperator
T[K]                                 tsindexedaccesstype


<a id="T08.04.3"></a>
## 08.04.3 · Where TypeScript5 holds a type

A type is a term of `TypeScript Types`, but TypeScript5's expressions hold one in a few places: `x as Reading` (a
`tsasexpression`), `x satisfies Reading` and `<Reading>x`, each its `typeAnnotation`; an arrow's type parameters,
parameters' annotations and return type; a call's type arguments. TypeScript5 reads `TypeScript Types`' terms, as each
dialect of the family reads the others' (`reads()`), and one printer prints them all (tutorial 8, 08.5), writing a
generic arrow's `<T,>` as a `.tsx` file needs it. Three of TypeScript5's kinds hold a type as a subterm, read from the
store; an annotation, a type parameter and a type argument hold one through TypeScript5's helpers:

In [4]:
identifier = lambda name: lambda e: e.identifier(lambda i: i.name(name))
with store.transaction():
    hosts = [
        S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.tsasexpression(lambda r: r
            .expression(identifier("x")).typeAnnotation(reference("Reading"))))),
        S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.tssatisfiesexpression(lambda r: r
            .expression(identifier("x")).typeAnnotation(reference("Reading"))))),
        S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.tstypeassertion(lambda r: r
            .typeAnnotation(reference("Reading")).expression(identifier("x"))))),
        S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.arrowfunctionexpression(lambda a: a
            .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
            .params([lambda p: p.identifier(lambda i: i.name("v").typeAnnotation(annotation(reference("T"))))])
            .returnType(annotation(lambda t: t.tsarraytype(lambda a: a.elementType(reference("T")))))
            .body(lambda b: b.callexpression(lambda c: c.callee(identifier("wrap"))
                .typeArguments(lambda g: g.params([reference("T")])).arguments([identifier("v")])))))),
    ]
for host in hosts:
    print(f"{TypeScript5.Text.ToText(host):36}", Terms.kind_of(Terms.term(host))[0])
types = store.registered(TYPES.union)
held = [f"{b.name}.{name}" for b in store.registered(TypeScript5.Expressions.DIALECT.union).branches
        for name, p in b.schema.properties.items() if types in (p.schema, getattr(p.schema, "element", None))]
print("TypeScript5's properties that hold a type:", ", ".join(held))
print("TypeScript5 reads", ", ".join(d.name() for d in TypeScript5.Expressions.DIALECT.reads()), "| one printer:",
      ToText is TypeScript5.Text.ToText)

x as Reading                         tsasexpression
x satisfies Reading                  tssatisfiesexpression
<Reading>x                           tstypeassertion
<T,>(v: T): T[] => wrap<T>(v)        arrowfunctionexpression
TypeScript5's properties that hold a type: tsasexpression.typeAnnotation, tssatisfiesexpression.typeAnnotation, tstypeassertion.typeAnnotation
TypeScript5 reads TypeScript Types, TypeScript Patterns, TypeScript RegExp | one printer: True


<a id="T08.04.4"></a>
## 08.04.4 · What a type binds, and the names it reads

A name in a type, `Reading` or `T`, names a type its program declares, or a value whose type `typeof` reads: something
no term models, so a type's names are references that validation never reports unbound, and that are never free. Three
kinds bind names within a type, and refine the framework's `Let` (tutorial 7): a mapped type binds its key, `K` in
`{ readonly [K in keyof T]: T[K] }`, its constraint Let's `value` and its type Let's `body`; a function type and a
constructor type bind their type parameters and parameters within them. A conditional type binds what its `infer`
declares, `V` in `T extends (infer V)[] ? V : never`, in its true branch. A name a value's place reads is TypeScript5's
`identifier`, which its scope gives, as in a computed key:

In [5]:
readonly = typed(lambda t: t.tsmappedtype(lambda m: m.readonly("readonly").key(lambda i: i.name("K"))
    .value(lambda v: v.tstypeoperator(lambda o: o.operator("keyof").typeAnnotation(reference("T"))))
    .body(lambda b: b.tsindexedaccesstype(lambda a: a.objectType(reference("T")).indexType(reference("K"))))))
element = typed(lambda t: t.tsconditionaltype(lambda c: c.checkType(reference("T"))
    .extendsType(lambda e: e.tsarraytype(lambda a: a.elementType(
        lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("V")))))))
    .trueType(reference("V")).falseType(keyword("tsneverkeyword"))))
computed = typed(lambda t: t.tstypeliteral(lambda o: o.members([lambda m: m.tspropertysignature(lambda s: s
    .computed(True).key(identifier("key")).typeAnnotation(annotation(keyword("tsstringkeyword"))))])))
for held in (readonly, element, typed(function), computed):
    kind = Terms.kind_of(Terms.term(held))
    print(f"{ToText(held):36}", f"{kind[0]:17}", "binder:", Terms.is_binder(kind[1].as_schema()), "| free:",
          ", ".join(sorted(Symbolics.free(held, TYPES))) or "none", "|", "; ".join(TYPES.validate(held)) or "valid")
form = Terms.form(Terms.term(readonly))
print("key:", form.attributes["key"].attributes["name"], "| value:", ToText(form.arguments[0]), "| body:", ToText(form.arguments[1]))

{ readonly [K in keyof T]: T[K] }    tsmappedtype      binder: True | free: none | valid
T extends (infer V)[] ? V : never    tsconditionaltype binder: False | free: none | valid
(x: number) => string                tsfunctiontype    binder: True | free: none | valid
{ [key]: string; }                   tstypeliteral     binder: False | free: key | members 0 key: identifier 'key' is not bound
key: K | value: keyof T | body: T[K]


<a id="T08.04.5"></a>
## 08.04.5 · Validated

Validation checks each kind as typescript-estree has it, in its own words: a union needs two types, a mapped type its
constraint, a conditional type all four of its parts, and a name, a reference's or a mapped type's key, must be one
TypeScript can spell. A refused type is
kept, and printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
refused = [
    lambda t: t.tsuniontype(lambda u: u.types([keyword("tsstringkeyword")])),
    lambda t: t.tsmappedtype(lambda m: m.key(lambda i: i.name("K K")).value(reference("T")).body(reference("T"))),
    lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("max-temp")))),
]
for spec in refused:
    held = typed(spec)
    print(f"{ToText(held):27}", "; ".join(TYPES.validate(held)))
assert TYPES.validate(typed(refused[0])) == ["a tsuniontype needs two types"]

string                      a tsuniontype needs two types
{ [K K in T]: T }           an identifier's name cannot be 'K K'
max-temp                    typeName: an identifier's name cannot be 'max-temp'


<a id="T08.04.6"></a>
## 08.04.6 · Inferred as any, and not simplified

A type describes values; it is not one. Inference gives a type the domain `any`, which says nothing, and TypeScript5's rules, which fold its literals alone (tutorial 3), fold none of `TypeScript Types`' kinds: each stays in the residual, named by the dialect, as an `as` stays, named by its own:

In [7]:
def refused_by(expression):
    """What TypeScript5's simplifier says of `expression`: its value, or why its residual is not a literal."""
    try:
        return TypeScript5.Simplifiers.OfAny(expression).evaluate()
    except NotEvaluated as error:
        return str(error)


union = typed(lambda t: t.tsuniontype(lambda u: u.types([keyword("tsstringkeyword"), keyword("tsnumberkeyword")])))
for held in (union, readonly):
    print(f"{ToText(held):36}", "inferred:", TYPES.infer(held).name(), "|", refused_by(held))
print(f"{TypeScript5.Text.ToText(hosts[0]):36}", refused_by(hosts[0]))

string | number                      inferred: any | the root is not a literal: TypeScript Types tsuniontype is not simplified
{ readonly [K in keyof T]: T[K] }    inferred: any | the root is not a literal: TypeScript Types tsmappedtype is not simplified
x as Reading                         the root is not a literal: TypeScript5 tsasexpression is not simplified


<a id="T08.04.7"></a>
## 08.04.7 · Its editions as data

`TypeScript Types` covers the versions of TypeScript 5 that TypeScript5 covers, 5.0 to 5.9, each found by its published
name, Microsoft's, and each with all 37 of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md));
the six terms its `Term` lists from TypeScript5 are recorded there:

In [8]:
dialect = store.Instances.Expressions.TypeScript.Types
editions = Standards.standards(dialect)
print(", ".join(editions))
print(Standards.properties(Standards.find(dialect, "TypeScript 5.4"))["bodies"][0], "| kinds in each:",
      ", ".join(str(len(Standards.kinds(edition))) for edition in editions.values()))

TypeScript 5.0, TypeScript 5.1, TypeScript 5.2, TypeScript 5.3, TypeScript 5.4, TypeScript 5.5, TypeScript 5.6, TypeScript 5.7, TypeScript 5.8, TypeScript 5.9
com.microsoft | kinds in each: 37, 37, 37, 37, 37, 37, 37, 37, 37, 37


## What you learned

- **A type is a term of its own dialect**, `TypeScript Types`: keywords, references, arrays, unions, literal, object,
  function, mapped and conditional types, 37 kinds, built from the smallest up.
- **TypeScript5 holds a type** after `as` and `satisfies`, in `<T>x`, and in a function's annotations and type
  arguments, and reads `TypeScript Types`' terms there; one printer writes both.
- **A type's names are references** to what its program declares, never reported unbound; a mapped type, a function
  type and a constructor type bind names within them, and refine `Let`.
- **Validation speaks typescript-estree's rules**; inference gives `any`, and simplification leaves each kind in the residual, named.
- **Its editions are data**: TypeScript 5.0 to 5.9.

Next, `TypeScript Patterns`: the destructuring a parameter or an assignment's target holds.

<!-- nav -->
[← 8.03 · Python RegExp](08_03_Python_RegExp.ipynb) · [Home](../../README.md) · [8.05 · TypeScript Patterns →](08_05_TypeScript_Patterns.ipynb)